In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [8]:
# Consolidate imports directly from starter
from starter import RAGBase, index, client, rag


query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop.

Each iteration:
1. Sends the full `messages` history to the model.
2. Checks the response for any `function_call` items.
3. Runs those tools and appends the outputs to `messages`.
4. If there were no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the response**.


### Here is what each line does:

**TracerProvider()** creates the SDK's central configuration object. It owns the span processors and decides how spans are built.  

**SimpleSpanProcessor(ConsoleSpanExporter())**  wires a processor that forwards every finished span to the console exporter, one at a time. "Simple" means synchronous and immediate - good for development.  

**trace.set_tracer_provider(provider)** registers the provider globally, so every call to trace.get_tracer(...) returns a tracer backed by it.  

**trace.get_tracer("llm-zoomcamp")** returns a Tracer we use to create spans. The string is just a label for the instrumentation scope - it identifies which part of the code produced the spans.  

In [6]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0x9ca41c68fbe33c2c4590002d4612dd93",
        "span_id": "0xa580a067d2108ce6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T23:20:11.754507Z",
    "end_time": "2026-07-20T23:20:11.754585Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0780c584-585c-413c-b27b-ea50da99beb5",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [20]:
# --- Traced Subclass ---
class RAGTraced(RAGBase):
    def __init__(self, tracer, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer

    def search(self, query: str):
        with self.tracer.start_as_current_span("search") as span:
            span.set_attribute("search.query", query)
            return super().search(query)

    def llm(self, prompt: str):
        with self.tracer.start_as_current_span("llm") as span:
            return super().llm(prompt)

    def rag(self, query: str):
        with self.tracer.start_as_current_span("rag") as span:
            span.set_attribute("rag.query", query)
            return super().rag(query)

if __name__ == "__main__":
    # --- OpenTelemetry Setup ---
    provider = TracerProvider()
    processor = SimpleSpanProcessor(ConsoleSpanExporter())
    provider.add_span_processor(processor)
    trace.set_tracer_provider(provider)
    tracer = trace.get_tracer("rag-tracer")

    # Instantiate the traced app using your real Zoomcamp index and OpenAI client
    traced_rag = RAGTraced(tracer=tracer, index=index, llm_client=client)

    # Run the query
    query = "How does the agentic loop keep calling the model until it stops?"
    print(f"Executing query: '{query}'\n")
    print("-" * 50)
    
    answer = traced_rag.rag(query)
    
    print("-" * 50)
    print(f"\nFinal LLM Response:\n{answer}")

Overriding of current TracerProvider is not allowed


Executing query: 'How does the agentic loop keep calling the model until it stops?'

--------------------------------------------------
{
    "name": "search",
    "context": {
        "trace_id": "0xdc9b865b85c0beb57ec65a00c3e5f890",
        "span_id": "0x0f7a8662ae480520",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc19646ca0ac9d859",
    "start_time": "2026-07-21T00:16:22.503202Z",
    "end_time": "2026-07-21T00:16:22.512799Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search.query": "How does the agentic loop keep calling the model until it stops?"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0780c584-585c-413c-b27b-ea50da99beb5",
            "service.name": "unknown_service"
     

In [15]:
class RAGTraced(RAGBase):
    def __init__(self, tracer, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer

    # ... search and rag methods remain the same ...

    def llm(self, prompt: str):
        with self.tracer.start_as_current_span("llm") as span:
            # Call the base class to get the raw response object
            response = super().llm(prompt)
            
            # Extract usage statistics
            usage = response.usage
            input_tokens = usage.input_tokens
            output_tokens = usage.output_tokens
            
            # Set tokens as attributes
            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)
            
            # Assuming gpt-4o-mini pricing: 
            # $0.150 per 1M input tokens, $0.600 per 1M output tokens
            cost = (input_tokens * 0.75 / 1_000_000) + (output_tokens * 4.5 / 1_000_000)
            span.set_attribute("cost", cost)
            
            return response

In [16]:
# Instantiate the traced app using your real Zoomcamp index and OpenAI client
traced_rag = RAGTraced(tracer=tracer, index=index, llm_client=client)

# Run the query
query = "How does the agentic loop keep calling the model until it stops?"
print(f"Executing query: '{query}'\n")
print("-" * 50)

answer = traced_rag.rag(query)

print("-" * 50)
print(f"\nFinal LLM Response:\n{answer}")

Executing query: 'How does the agentic loop keep calling the model until it stops?'

--------------------------------------------------
{
    "name": "llm",
    "context": {
        "trace_id": "0x1795f2c3873932161803ff707afef59a",
        "span_id": "0xac2d6837d36716ce",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T23:56:49.612872Z",
    "end_time": "2026-07-20T23:56:58.700726Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 96,
        "cost": 0.00576525
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0780c584-585c-413c-b27b-ea50da99beb5",
            "service.name": "unknown_service"
        },
        "schema_ur

### Q1. First trace
Wrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of RAGBase that wraps rag(), search(), and llm() each in their own span.  

Run this query:   

How does the agentic loop keep calling the model until it stops?  

The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?  

1  
**3**  
5  
7  
## Q2. Capturing metrics as span attributes
Spans are not just timing markers - you can attach any information you want to them with set_attribute. We already use spans to record how long each step takes. Now we'll add the metrics we care about: tokens and cost.

Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span:

span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
And since we know both input and output tokens, we can also compute the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

700  
**7000**  
70000  
700000  
These numbers vary between runs. Pick the closest option.

In [17]:
from datetime import datetime

# Copy the start and end times from your console output
start_time_str = "2026-07-20T23:29:13.059696Z"
end_time_str = "2026-07-20T23:29:16.950294Z"

# Replace 'Z' with '+00:00' so Python's datetime can parse the string easily
start_time = datetime.fromisoformat(start_time_str.replace("Z", "+00:00"))
end_time = datetime.fromisoformat(end_time_str.replace("Z", "+00:00"))

# Calculate the difference
duration = end_time - start_time

# Convert to milliseconds
duration_ms = duration.total_seconds() * 1000

print(f"LLM Span Duration: {duration_ms:.2f} ms")

LLM Span Duration: 3890.60 ms


## Q3. Span timing
Each span automatically records its duration. Look at the console output from Q1 and find the durations for the search span and the llm span.  

For a typical query, roughly how long does the LLM call take?  

Under 100ms  
100-500ms  
500-2000ms   
**Over 2000ms**  
The first call can be slower (cold start). Pick the range you see most often.



In [18]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [19]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [23]:
import sqlite3
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult

# Import your setup from starter.py
from starter import RAGBase, index, client

# --- 1. Custom SQLite Exporter ---
class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

# --- 2. Traced RAG Class ---
class RAGTraced(RAGBase):
    def __init__(self, tracer, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer

    def search(self, query: str):
        with self.tracer.start_as_current_span("search") as span:
            span.set_attribute("search.query", query)
            return super().search(query)

    def llm(self, prompt: str):
        with self.tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            
            # Extract usage statistics
            usage = response.usage
            input_tokens = usage.input_tokens
            output_tokens = usage.output_tokens
            
            # Set tokens and cost as attributes
            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)
            
            cost = (input_tokens * 0.150 / 1_000_000) + (output_tokens * 0.600 / 1_000_000)
            span.set_attribute("cost", cost)
            
            return response

    def rag(self, query: str):
        with self.tracer.start_as_current_span("rag") as span:
            span.set_attribute("rag.query", query)
            return super().rag(query)

# --- 3. Execution & Verification ---
if __name__ == "__main__":
    # Initialize the TracerProvider
    provider = TracerProvider()
    
    # Use our custom SQLite Span Exporter instead of the Console Exporter
    processor = SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
    provider.add_span_processor(processor)
    
    
    tracer = trace.get_tracer("rag-tracer")
    tracer = trace.get_tracer("rag-tracer", tracer_provider=provider)

    # Instantiate the traced app
    traced_rag = RAGTraced(tracer=tracer, index=index, llm_client=client)

    # Run the query
    query = "How does the agentic loop keep calling the model until it stops?"
    print(f"Executing query: '{query}'\n")
    answer = traced_rag.rag(query)
    print("Query finished!\n")

    # Check the database contents
    print("-" * 50)
    print("Querying traces.db for saved spans:")
    print("-" * 50)
    
    conn = sqlite3.connect("traces.db")
    cursor = conn.cursor()
    cursor.execute("SELECT name, input_tokens, output_tokens, cost FROM spans;")
    rows = cursor.fetchall()
    
    for row in rows:
        name, in_tokens, out_tokens, cost = row
        print(f"Span Name: {name}")
        if in_tokens is not None:
             print(f"  Input Tokens:  {in_tokens}")
             print(f"  Output Tokens: {out_tokens}")
             print(f"  Cost:          ${cost:.6f}")
        print()
        
    conn.close()

Executing query: 'How does the agentic loop keep calling the model until it stops?'

Query finished!

--------------------------------------------------
Querying traces.db for saved spans:
--------------------------------------------------
Span Name: search

Span Name: llm
  Input Tokens:  7111
  Output Tokens: 90
  Cost:          $0.001121

Span Name: rag



In [ ]:
if __name__ == "__main__":
    # --- OpenTelemetry Setup ---
    provider = TracerProvider()
    processor = SimpleSpanProcessor(ConsoleSpanExporter())
    provider.add_span_processor(processor)
    trace.set_tracer_provider(provider)
    tracer = trace.get_tracer("rag-tracer")

    # Instantiate the traced app using your real Zoomcamp index and OpenAI client
    traced_rag = RAGTraced(tracer=tracer, index=index, llm_client=client)

    # Run the query
    query = "How does the agentic loop keep calling the model until it stops?"
    print(f"Executing query: '{query}'\n")
    print("-" * 50)
    
    answer = traced_rag.rag(query)
    
    print("-" * 50)
    print(f"\nFinal LLM Response:\n{answer}")

## Q4. Saving traces to SQLite
Re-run the query from Q1. Which span names appear in the spans table?  

Only rag  
rag and llm  
**rag, search, and llm**  
search, llm, and judge  

In [24]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect("traces.db")

# Query to calculate total duration in seconds (OTel timestamps are in nanoseconds)
query = """
    SELECT 
        name, 
        SUM(end_time - start_time) / 1e9 AS total_duration_seconds 
    FROM spans 
    WHERE name != 'rag' 
    GROUP BY name
    ORDER BY total_duration_seconds DESC;
"""

# Execute the query and load into a DataFrame
df = pd.read_sql_query(query, conn)

print(df)
conn.close()

     name  total_duration_seconds
0     llm                1.603970
1  search                0.005575


## Q5. Querying trace data
The traces are now in SQLite. Run one more query through the traced RAG, then query the database.  

The rag span wraps everything, so its duration includes both search and llm. To see where time actually goes, exclude the rag span and compare the   children.  

Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?  

search  
**llm**   
They're all about the same  

In [25]:
# 1. Run the same query 3 more times
query = "How does the agentic loop keep calling the model until it stops?"
for i in range(3):
    print(f"Running iteration {i+1}...")
    traced_rag.rag(query)

# 2. Query the database using pandas
conn = sqlite3.connect("traces.db")

sql = """
    SELECT start_time, input_tokens 
    FROM spans 
    WHERE name = 'llm'
"""

df = pd.read_sql_query(sql, conn)
print("\nLLM Span Input Tokens:")
print(df)

conn.close()

Running iteration 1...
Running iteration 2...
Running iteration 3...

LLM Span Input Tokens:
            start_time  input_tokens
0  1784593542565635500          7111
1  1784593925369916500          7111
2  1784593928880345400          7111
3  1784593931170609000          7111


## Q6. Token stability across runs   
Load the SQLite data with pandas. One thing a dashboard can tell you is how stable your system is. If the same query always produces the same number of input tokens, the context your RAG retrieves is consistent. If it varies a lot, something in the search may be unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span.  

How much do the input tokens vary across these 4 runs?  

**They're identical**  
Within 10% of each other  
Within 50% of each other  
They vary more than 50%  